# 1. 같은 배치를 새 ID로도 비교할 수 있을까요?

기존 모델은 Train에서 본 배치 ID만 출력합니다. 후보 scorer는 각 선택지에 같은 함수를 적용할 수 있다는 가설입니다. **이번 Notebook은 그 전에 후보 목록에 Human 정답이 얼마나 포함되는지 확인합니다. 학습 코드는 아직 구현하지 않았습니다.**

[실험 명세](../../../ai-harness/ml-experiments/experiments/human-placement-candidate-scoring.md)를 따릅니다. 모듈형/통합 신경망 채택이나 gameplay 평가는 아닙니다. 별도 Python 파일을 실행할 필요가 없습니다.

전체 실행의 기본값은 작은 내부 assert뿐입니다. 실제 자료는 4절에서 직접 선택하며 모델 Full은 coverage 검토 전 차단합니다.


## 2. 왜 최종 좌표를 사용하지 않나요?

빈 20×10 보드에 square를 놓는 정적 후보는 바닥의 가로 위치 9개입니다. 회전 ID 4개는 같은 셀을 차지하므로 class마다 alias 4개로 묶습니다. Human 정답이 4번째 후보라면 coverage=1, 후보에 없는 셀이라면 coverage=0입니다. 정답을 후보에 추가하지 않습니다.

이 후보는 **충돌하지 않고 한 칸 아래로 내려갈 수 없는 셀 배치**입니다. 출발점에서 들어갈 수 없는 구멍도 포함할 수 있습니다. 최종 activePiece를 출발점으로 삼거나 spawn을 추정하지 않으며 실제 reachability·game-over 안전성·historical ruleset은 미검증입니다.


In [ ]:
import ast
import copy
import hashlib
import json
import os
import platform
import re
import shutil
import subprocess
import tempfile
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

os.environ["HUMAN_CANDIDATE_NODE"] = (
    "/Users/parkseongho/.nvm/versions/node/v24.18.0/bin/node"
)

print(subprocess.check_output(
    [os.environ["HUMAN_CANDIDATE_NODE"], "--version"],
    text=True,
).strip())

ROOT = next(p.resolve() for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ai-harness").is_dir() and (p / "package.json").is_file())
EXPERIMENT_ID = "human-placement-candidate-scoring"
SOURCE_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_placement_candidate_scoring.ipynb"
SNAPSHOT_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_placement_snapshot_baseline.ipynb"
SNAPSHOT_DEFS_SHA = "233ceca8322bd4335b798cdd452064d389e0ddbce6451610c7ba7d56a1dd9c5c"
CONTRACT = "snapshot_static_supported_cells_v1"
selected = [{"id": c["id"], "source": "".join(c["source"])}
            for c in json.loads(SNAPSHOT_NOTEBOOK.read_text(encoding="utf-8"))["cells"]
            if c["cell_type"] == "code" and "definitions" in c.get("metadata", {}).get("tags", [])]
assert hashlib.sha256((json.dumps(selected, ensure_ascii=False, sort_keys=True,
                                  allow_nan=False, indent=2) + "\n").encode()).hexdigest() == SNAPSHOT_DEFS_SHA
shared = {"__name__": "candidate_coverage_shared"}
for cell in selected:
    exec(compile(cell["source"], str(SNAPSHOT_NOTEBOOK), "exec"), shared)
shared["EXPERIMENT_ID"], shared["SOURCE_NOTEBOOK"] = EXPERIMENT_ID, SOURCE_NOTEBOOK
shared["base"]["dependency"]["link"]["dataset"]["shared"]["EXPERIMENT_ID"] = EXPERIMENT_ID
for name in ("require", "json_bytes", "sha256_file", "definition_sha", "physical_key",
             "load_snapshots", "report_path", "publish_report", "reject_constant", "finite_float"):
    globals()[name] = shared[name]
MANIFEST_SHA = shared["MANIFEST_SHA"]
EXPECTED = shared["EXPECTED"]

BASELINE_VOCAB_COUNT = 2133
BASELINE_VOCAB_SHA = "26edffc2ac5501450f78e976818cba195882d558c41530ff2196a0b017d45084"
BASELINE_OOV = {"train": 0, "validation": 152}


In [ ]:
CORE_ENTRY = ROOT / "src/client/game/ai/heuristic/placementGenerator.ts"
BLOCKS_ENTRY = ROOT / "src/client/game/state/blocks.ts"

def core_fingerprints():
    """Fingerprint relative TS/MJS dependency closure, including type-only imports conservatively."""
    pending, found = [CORE_ENTRY, BLOCKS_ENTRY], {}
    while pending:
        path = pending.pop().resolve()
        require(path.is_relative_to(ROOT) and path.is_file(), "CORE_DEPENDENCY_MISSING")
        relative = path.relative_to(ROOT).as_posix()
        if relative in found:
            continue
        found[relative] = sha256_file(path)
        for target in re.findall(r"(?:from|import)\s*[\"']([^\"']+)[\"']", path.read_text(encoding="utf-8")):
            if target.startswith("."):
                pending.append(path.parent / target)
    return dict(sorted(found.items()))

CORE_EXPECTED = core_fingerprints()
NODE_EXECUTABLE = os.environ.get("HUMAN_CANDIDATE_NODE", "node")
# Only geometry enumeration lives in JS; no Human target, heuristic score or learning is passed here.
CORE_BRIDGE = r"""
import { createInterface } from "node:readline";
const { canPlacePlacement, normalizePlacement } = await import(process.env.CANDIDATE_PLACEMENT_MODULE);
const { default: BLOCKS } = await import(process.env.CANDIDATE_BLOCKS_MODULE);
const check = (ok, reason) => { if (!ok) throw new Error(reason); };
const compare = (a, b) => a < b ? -1 : a > b ? 1 : 0;
for await (const line of createInterface({ input: process.stdin, crlfDelay: Infinity })) {
    const request = JSON.parse(line);
    check(Object.keys(request).sort().join(",") === "board,pieceType", "REQUEST_FIELDS_INVALID");
    const { board, pieceType } = request;
    check(Array.isArray(board) && board.length === 20 && board.every(row =>
        Array.isArray(row) && row.length === 10 && row.every(c => c === null || c === "occupied")),
        "BOARD_INVALID");
    check(typeof pieceType === "string" && Object.hasOwn(BLOCKS, pieceType), "PIECE_INVALID");
    const before = JSON.stringify(board);
    const classes = new Map();
    for (let rotation = 0; rotation < BLOCKS[pieceType].length; rotation++) {
        const offsets = BLOCKS[pieceType][rotation];
        const xs = offsets.map(c => c[0]), ys = offsets.map(c => c[1]);
        for (let row = -Math.min(...ys); row <= 19 - Math.max(...ys); row++) {
            for (let col = -Math.min(...xs); col <= 9 - Math.max(...xs); col++) {
                if (!canPlacePlacement(board, pieceType, rotation, col, row) ||
                    canPlacePlacement(board, pieceType, rotation, col, row + 1)) continue;
                const placement = normalizePlacement({ pieceType, rotation, col, row });
                const cells = placement.cells.sort((a, b) => a.row - b.row || a.col - b.col);
                const key = pieceType + "|" + cells.map(c => c.row + "," + c.col).join(";");
                if (classes.has(key)) classes.get(key).aliasCount++;
                else classes.set(key, { key, cells, aliasCount: 1,
                    representative: { rotation, row, col } });
            }
        }
    }
    check(before === JSON.stringify(board), "BOARD_MUTATED");
    process.stdout.write(JSON.stringify([...classes.values()].sort((a,b) => compare(a.key,b.key))) + "\n");
}
"""

def candidate_requests(rows):
    """Allowlist board occupancy and pieceType only; never pass target/final pose/IDs to core."""
    requests = []
    for row in rows:
        x = np.asarray(row["x"])
        require(x.shape == (207,) and np.isfinite(x).all()
                and np.isin(x, [0, 1]).all(), "INPUT_FEATURES_INVALID")
        require(row["pieceType"] in shared["PIECES"]
                and x[200:].tolist() == [float(row["pieceType"] == p) for p in shared["PIECES"]],
                "PIECE_FEATURE_MISMATCH")
        board = [[None if x[r * 10 + c] == 0 else "occupied" for c in range(10)] for r in range(20)]
        requests.append({"board": board, "pieceType": row["pieceType"]})
    return requests

def core_candidates(rows):
    """Bound one process to at most64 snapshots; validate Node24 and response format."""
    require(0 < len(rows) <= 64, "CORE_BATCH_SIZE_INVALID")
    require(core_fingerprints() == CORE_EXPECTED, "CORE_CHANGED")
    executable = shutil.which(NODE_EXECUTABLE)
    require(executable is not None, "NODE24_REQUIRED_SET_HUMAN_CANDIDATE_NODE")
    version = subprocess.run([executable, "--version"], capture_output=True, text=True,
                             check=True, timeout=10).stdout.strip()
    require(re.fullmatch(r"v24\.(\d+)\.(\d+)", version) is not None
            and tuple(map(int, version[1:].split("."))) >= (24, 18, 0), "NODE24_18_REQUIRED")
    env = {**os.environ, "CANDIDATE_PLACEMENT_MODULE": CORE_ENTRY.as_uri(),
           "CANDIDATE_BLOCKS_MODULE": BLOCKS_ENTRY.as_uri()}
    payload = "".join(json.dumps(r, allow_nan=False, separators=(",", ":")) + "\n"
                      for r in candidate_requests(rows))
    result = subprocess.run([executable, "--input-type=module", "-e", CORE_BRIDGE],
                            input=payload, text=True, capture_output=True, cwd=ROOT,
                            env=env, timeout=60)
    require(result.returncode == 0, "CORE_PROCESS_FAILED_CHECK_NODE_AND_SOURCE")
    lines = result.stdout.splitlines()
    require(len(lines) == len(rows), "CORE_RESPONSE_COUNT_INVALID")
    batches = [json.loads(line, parse_constant=reject_constant, parse_float=finite_float) for line in lines]
    for row, candidates in zip(rows, batches):
        require(isinstance(candidates, list) and len(candidates) <= 800, "CORE_RESPONSE_INVALID")
        keys = []
        for candidate in candidates:
            require(set(candidate) == {"key", "cells", "aliasCount", "representative"}
                    and type(candidate["aliasCount"]) is int and 1 <= candidate["aliasCount"] <= 4,
                    "CORE_CLASS_INVALID")
            pose = candidate["representative"]
            key = physical_key({"pieceType": row["pieceType"], **pose, "cells": candidate["cells"]}, selected=True)
            require(candidate["key"] == key, "CORE_KEY_INVALID")
            keys.append(key)
        require(keys == sorted(set(keys)), "CORE_ORDER_OR_DUPLICATE_INVALID")
    require(core_fingerprints() == CORE_EXPECTED, "CORE_CHANGED")
    return batches, version


In [ ]:
def coverage_record(row, candidates, vocabulary):
    """Match Human label only AFTER target-independent candidate generation."""
    keys = [c["key"] for c in candidates]
    matched = row["targetKey"] in keys
    count = len(keys)
    return {"segmentId": row["segmentId"], "gameId": row["gameId"], "split": row["split"],
            "pieceType": row["pieceType"], "sourceLine": row["sourceLine"],
            "baselineTargetOov": row["targetKey"] not in vocabulary,
            "candidates": count, "rawAliases": sum(c["aliasCount"] for c in candidates),
            "targetIncluded": matched, "choiceEligible": matched and count >= 2,
            "candidateSetSha256": hashlib.sha256(json_bytes(candidates)).hexdigest(),
            "reason": "no_candidates" if count == 0 else
                      "target_not_in_static_set" if not matched else
                      "singleton_target_included" if count == 1 else "choice_required_target_included"}

def summarize_coverage(records):
    require(records and len({r["segmentId"] for r in records}) == len(records), "EMPTY_OR_DUPLICATE_COHORT")
    return {"games": len({r["gameId"] for r in records}), "segments": len(records),
            "targetIncluded": sum(r["targetIncluded"] for r in records),
            "targetCoverage": sum(r["targetIncluded"] for r in records) / len(records),
            "choiceEligible": sum(r["choiceEligible"] for r in records),
            "noCandidates": sum(r["candidates"] == 0 for r in records),
            "singleton": sum(r["candidates"] == 1 for r in records),
            "candidateClasses": sum(r["candidates"] for r in records),
            "rawAliases": sum(r["rawAliases"] for r in records),
            "baselineOov": sum(r["baselineTargetOov"] for r in records),
            "reasons": dict(sorted(Counter(r["reason"] for r in records).items()))}

def grouped_coverage(records, fields):
    groups = defaultdict(list)
    for row in records:
        groups[tuple(row[f] for f in fields)].append(row)
    return [{**dict(zip(fields, key)), **summarize_coverage(rows)}
            for key, rows in sorted(groups.items())]

def coverage_paths(report_name, evidence_name):
    report, evidence = report_path(report_name), report_path(evidence_name)
    require(report != evidence, "OUTPUT_PATH_COLLISION")
    return report, evidence

def run_coverage(report_name, evidence_name):
    """Full data is opened only here. Publish evidence first, report last; never overwrite."""
    report_path_value, evidence_path = coverage_paths(report_name, evidence_name)
    source_sha = definition_sha(SOURCE_NOTEBOOK)
    require(core_fingerprints() == CORE_EXPECTED, "CORE_CHANGED")
    data, watched = load_snapshots()
    vocabulary = set(shared["train_vocabulary"](data["train"]))
    require(len(vocabulary) == BASELINE_VOCAB_COUNT and hashlib.sha256(json_bytes(sorted(vocabulary))).hexdigest()
            == BASELINE_VOCAB_SHA,
            "TRAIN_VOCABULARY_CHANGED")
    records, node_version, temporary = [], None, None
    try:
        with tempfile.NamedTemporaryFile(dir=evidence_path.parent, prefix=".candidate-evidence-", delete=False) as handle:
            temporary = Path(handle.name)
            handle.write(b"[\n")
            first = True
            for split in ("train", "validation"):
                rows = data[split]
                for start in range(0, len(rows), 64):
                    batch = rows[start:start + 64]
                    outputs, version = core_candidates(batch)
                    require(node_version is None or node_version == version, "NODE_VERSION_CHANGED")
                    node_version = version
                    for row, candidates in zip(batch, outputs):
                        record = coverage_record(row, candidates, vocabulary)
                        records.append(record)
                        if not first:
                            handle.write(b",\n")
                        handle.write(json_bytes({"segmentId": row["segmentId"], "split": split,
                                                 "candidates": candidates}).rstrip(b"\n"))
                        first = False
            handle.write(b"\n]\n")
            handle.flush()
            os.fsync(handle.fileno())
        require(len(records) == sum(v[1] for v in EXPECTED.values()) and all(sha256_file(p) == sha for p, sha in watched.items())
                and core_fingerprints() == CORE_EXPECTED and definition_sha(SOURCE_NOTEBOOK) == source_sha
                and definition_sha(SNAPSHOT_NOTEBOOK) == SNAPSHOT_DEFS_SHA, "INPUT_OR_CODE_CHANGED")
        report = {"reportSchemaVersion": 1, "experimentId": EXPERIMENT_ID, "runMode": "full",
                  "executionPolicy": "full-only", "analysisStatus": "candidate_coverage_requires_review",
                  "candidateContract": CONTRACT, "fullCohortCovered": True, "policyDecisionEligible": False,
                  "canonicalResultEligible": False, "productionIntegration": False, "inferenceEligible": False,
                  "wholePolicyEvidence": False, "historicalRulesetVerified": False,
                  "reachableCandidatesGenerated": False, "gameOverSafetyChecked": False,
                  "staticGeometryChecked": True, "modelFitCount": 0, "scalerFitCount": 0,
                  "testFileRead": False, "rawFileRead": False, "validationPayloadRead": True,
                  "dataModified": False, "dataExcludedByAudit": 0, "gameIdLeakage": 0,
                  "summary": grouped_coverage(records, ["split"]),
                  "byPiece": grouped_coverage(records, ["split", "pieceType"]),
                  "byGame": grouped_coverage(records, ["split", "gameId"]),
                  "byBaselineOov": grouped_coverage(records, ["split", "baselineTargetOov"]),
                  "segments": records, "baselineVocabularySha256": hashlib.sha256(json_bytes(sorted(vocabulary))).hexdigest(),
                  "provenance": {"manifestSha256": MANIFEST_SHA,
                      "datasetSha256": {s: EXPECTED[s][-1] for s in EXPECTED},
                      "sourceDefinitionSha256": source_sha, "snapshotDefinitionSha256": SNAPSHOT_DEFS_SHA,
                      "coreFiles": CORE_EXPECTED},
                  "evidence": {"sha256": sha256_file(temporary), "segments": len(records)},
                  "environment": {"python": platform.python_version(), "numpy": np.__version__,
                                  "pandas": pd.__version__, "node": node_version},
                  "limits": ["snapshot_association_not_action_time_observation",
                             "static_support_not_spawn_reachability", "target_absence_not_automatic_bad_label",
                             "baseline_oov_uses_original_train_not_future_candidate_fit_vocabulary"]}
        os.link(temporary, evidence_path)
        # Report is the completion marker. A publication race may leave owned evidence but never overwrite files.
        publish_report(report, report_path_value)
        return report
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)


In [ ]:
def check_coverage_report(report):
    """Reject stale/other reports; reconstruct displayed summaries from per-segment diagnostics."""
    require(report["reportSchemaVersion"] == 1 and report["experimentId"] == EXPERIMENT_ID
            and report["runMode"] == "full" and report["candidateContract"] == CONTRACT
            and report["analysisStatus"] == "candidate_coverage_requires_review"
            and report["fullCohortCovered"] is True and report["staticGeometryChecked"] is True,
            "REPORT_CONTRACT_INVALID")
    for field in ("policyDecisionEligible", "canonicalResultEligible", "productionIntegration",
                  "inferenceEligible", "wholePolicyEvidence", "historicalRulesetVerified",
                  "reachableCandidatesGenerated", "gameOverSafetyChecked", "testFileRead",
                  "rawFileRead", "dataModified"):
        require(report[field] is False, "REPORT_FLAGS_INVALID")
    require(report["validationPayloadRead"] is True and report["modelFitCount"] == 0
            and report["scalerFitCount"] == 0 and report["dataExcludedByAudit"] == 0
            and report["gameIdLeakage"] == 0, "REPORT_SAFETY_INVALID")
    require(report["provenance"] == {"manifestSha256": MANIFEST_SHA,
        "datasetSha256": {s: EXPECTED[s][-1] for s in EXPECTED},
        "sourceDefinitionSha256": definition_sha(SOURCE_NOTEBOOK),
        "snapshotDefinitionSha256": SNAPSHOT_DEFS_SHA, "coreFiles": core_fingerprints()},
        "STALE_SOURCE_OR_DATASET_REPORT")
    require(definition_sha(SNAPSHOT_NOTEBOOK) == SNAPSHOT_DEFS_SHA
            and report["baselineVocabularySha256"] == BASELINE_VOCAB_SHA, "BASELINE_DEFINITIONS_CHANGED")
    records = report["segments"]
    require(len(records) == sum(v[1] for v in EXPECTED.values()) and len({r["segmentId"] for r in records}) == len(records),
            "REPORT_COHORT_INVALID")
    for row in records:
        require(row["split"] in EXPECTED and row["pieceType"] in shared["PIECES"]
                and type(row["candidates"]) is int and 0 <= row["candidates"] <= 800
                and type(row["rawAliases"]) is int
                and row["candidates"] <= row["rawAliases"] <= row["candidates"] * 4
                and all(type(row[f]) is bool for f in ("targetIncluded", "choiceEligible", "baselineTargetOov"))
                and row["choiceEligible"] == (row["targetIncluded"] and row["candidates"] >= 2)
                and (not row["targetIncluded"] or row["candidates"] > 0)
                and re.fullmatch(r"[a-f0-9]{64}", row["candidateSetSha256"]) is not None,
                "REPORT_SEGMENT_INVALID")
        expected_reason = ("no_candidates" if row["candidates"] == 0 else
            "target_not_in_static_set" if not row["targetIncluded"] else
            "singleton_target_included" if row["candidates"] == 1 else "choice_required_target_included")
        require(row["reason"] == expected_reason, "REPORT_REASON_INVALID")
    for split, expected in EXPECTED.items():
        selected_rows = [r for r in records if r["split"] == split]
        require(len(selected_rows) == expected[1]
                and len({r["gameId"] for r in selected_rows}) == expected[2]
                and sum(r["baselineTargetOov"] for r in selected_rows) == BASELINE_OOV[split],
                "REPORT_SPLIT_INVALID")
    require(not {r["gameId"] for r in records if r["split"] == "train"} &
            {r["gameId"] for r in records if r["split"] == "validation"}, "REPORT_SPLIT_LEAKAGE")
    for key, fields in (("summary", ["split"]), ("byPiece", ["split", "pieceType"]),
                        ("byGame", ["split", "gameId"]), ("byBaselineOov", ["split", "baselineTargetOov"])):
        require(report[key] == grouped_coverage(records, fields), "REPORT_AGGREGATION_INVALID")
    require(report["evidence"]["segments"] == len(records)
            and re.fullmatch(r"[a-f0-9]{64}", report["evidence"]["sha256"]) is not None,
            "EVIDENCE_CONTRACT_INVALID")
    return report

def show_coverage(report):
    check_coverage_report(report)
    print("Full candidate coverage; model/scaler fit=0; NOT reachability or gameplay evidence")
    display(pd.DataFrame(report["summary"]).set_index("split"))
    display(pd.DataFrame(report["byPiece"]))
    display(pd.DataFrame(report["byBaselineOov"]))
    display(pd.DataFrame(report["byGame"]).sort_values(["targetCoverage", "segments"]))
    per_segment = pd.DataFrame(report["segments"])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for split, rows in per_segment.groupby("split", sort=True):
        values = rows["candidates"].value_counts().sort_index()
        axes[0].plot(values.index, values.values, marker=".", label=split)
    axes[0].set(xlabel="Distinct static candidate classes", ylabel="Segments (all eligible)",
                title="Candidate count, not reachable-path count")
    axes[0].legend()
    frame = pd.DataFrame(report["byPiece"])
    frame.pivot(index="pieceType", columns="split", values="targetCoverage").plot.bar(ax=axes[1])
    axes[1].set(ylabel="Human target included / all eligible", ylim=(0, 1),
                title="Static target coverage; not model accuracy")
    fig.tight_layout()
    plt.show()
    plt.close(fig)


## 3. 작은 내부 검사 — 모델 학습이 아닙니다

Node24.18 이상/25 미만이 필요합니다. 커널에서 다른 Node가 잡히면 `HUMAN_CANDIDATE_NODE`에 사용할 Node 실행 파일 경로를 지정하고 커널을 재시작하세요. 아래 검사는 메모리 가상 보드만 사용합니다.


In [ ]:
def internal_checks():
    def fixture(piece="square"):
        return {"segmentId": "fixture", "gameId": "synthetic-game", "split": "train",
                "sourceLine": 1, "pieceType": piece,
                "x": np.array([0.] * 200 + [float(piece == p) for p in shared["PIECES"]]),
                "targetKey": "not_in_candidates", "targetCells": []}
    row = fixture()
    before = copy.deepcopy(row)
    first, _ = core_candidates([row])
    candidates = first[0]
    assert len(candidates) == 9 and all(c["aliasCount"] == 4 for c in candidates)
    assert all({c["row"] for c in item["cells"]} == {18, 19} for item in candidates)
    row["targetKey"], row["targetCells"] = candidates[3]["key"], candidates[3]["cells"]
    assert candidate_requests([row]) == candidate_requests([before])
    assert core_candidates([row])[0] == first
    assert np.array_equal(before["x"], row["x"])
    record = coverage_record(row, candidates, {row["targetKey"]})
    assert record["targetIncluded"] and record["choiceEligible"] and not record["baselineTargetOov"]
    assert record["rawAliases"] == 36
    missing = coverage_record(before, candidates, set())
    assert not missing["targetIncluded"] and missing["baselineTargetOov"]
    assert summarize_coverage([record])["targetCoverage"] == 1
    assert coverage_record(row, [], set())["reason"] == "no_candidates"
    assert coverage_record(row, [candidates[3]], set())["reason"] == "singleton_target_included"
    blocked = fixture()
    blocked["x"][:200] = 1
    assert core_candidates([blocked])[0] == [[]]
    bar = fixture("bar")
    assert len(core_candidates([bar])[0][0]) == 17
    for bad in (np.full(207, np.nan), np.full(207, 2), np.zeros(206)):
        invalid = {**row, "x": bad}
        try:
            candidate_requests([invalid])
            raise AssertionError("invalid input accepted")
        except ValueError:
            pass
    with tempfile.TemporaryDirectory(prefix="candidate-coverage-check-") as directory:
        root = Path(directory)
        existing = root / "keep.json"
        existing.write_text("keep", encoding="utf-8")
        link, hard, dangling = root / "link.json", root / "hard.json", root / "dangling.json"
        link.symlink_to(existing)
        os.link(existing, hard)
        dangling.symlink_to(root / "missing.json")
        for destination in (existing, link, hard, dangling, root / ".." / root.name / "alias.json", SOURCE_NOTEBOOK):
            try:
                report_path(str(destination))
                raise AssertionError("unsafe output accepted")
            except ValueError:
                pass
        destination = root / "new.json"
        publish_report({"synthetic": True}, destination)
        try:
            publish_report({"overwrite": True}, destination)
            raise AssertionError("existing publication accepted")
        except ValueError:
            pass
        assert json.loads(destination.read_text()) == {"synthetic": True}
        assert existing.read_text() == "keep"
    print("PASS: core empty/full board, square aliases, bar classes, label independence, bad inputs, denominators, output guards")

internal_checks()


In [ ]:
def publication_checks():
    from unittest.mock import patch
    # Three synthetic records, not a Human Full: exercise the same publication path with no fit.
    with tempfile.TemporaryDirectory(dir="/private/tmp", prefix="candidate-publication-") as directory:
        root = Path(directory)
        row = {"pieceType": "square", "sourceLine": 1,
               "x": np.array([0.] * 200 + [float("square" == p) for p in shared["PIECES"]])}
        candidates = core_candidates([row])[0][0]
        train = [{**row, "split": "train", "segmentId": f"t{i}", "gameId": f"train{i}",
                  "targetKey": candidates[i]["key"]} for i in range(2)]
        val = [{**row, "split": "validation", "segmentId": "v", "gameId": "val",
                "targetKey": candidates[2]["key"]}]
        keys = sorted(r["targetKey"] for r in train)
        with patch.dict(globals(), {"EXPECTED": {"train": (2, 2, 2, "fixture"),
                        "validation": (1, 1, 1, "fixture")}, "BASELINE_VOCAB_COUNT": 2,
                        "BASELINE_VOCAB_SHA": hashlib.sha256(json_bytes(keys)).hexdigest(),
                        "BASELINE_OOV": {"train": 0, "validation": 1},
                        "load_snapshots": lambda: ({"train": train, "validation": val}, {})}):
            report = run_coverage(str(root / "report.json"), str(root / "evidence.json"))
            check_coverage_report(report)
            assert len(json.loads((root / "evidence.json").read_text())) == 3
            assert sha256_file(root / "evidence.json") == report["evidence"]["sha256"]
            second = run_coverage(str(root / "report2.json"), str(root / "evidence2.json"))
            assert (root / "evidence.json").read_bytes() == (root / "evidence2.json").read_bytes()
            assert json_bytes(report) == json_bytes(second)
            for field in ("modelFitCount", "policyDecisionEligible", "reachableCandidatesGenerated"):
                changed = copy.deepcopy(report)
                changed[field] = 1 if field == "modelFitCount" else True
                try:
                    check_coverage_report(changed)
                    raise AssertionError("tampered report accepted")
                except ValueError:
                    pass
    print("PASS: synthetic publication, evidence SHA, deterministic bytes, tamper rejection; fit=0")

publication_checks()


## 4. 후보 지원도 Full — 학습 없음, 직접 실행 선택

설정 → 실행 → 결과 순서입니다. 처음에는 `RUN_COVERAGE_FULL=False`로 두세요. 실행할 때만 True로 바꾸고 두 출력 경로를 아직 없는 이름으로 정합니다. 기존 파일은 덮어쓰지 않습니다.

Evidence는 candidate 목록의 JSON 배열이고 report는 요약/구간 진단입니다. Evidence 먼저·report 마지막에 독점 게시합니다. 게시 경쟁/실패로 evidence만 남으면 완료가 아니며 두 경로를 새 이름으로 바꿔 재시도합니다. 기존 evidence를 자동 삭제하지 않습니다.


In [ ]:
RUN_COVERAGE_FULL = False
COVERAGE_REPORT_PATH = os.environ.get("HUMAN_CANDIDATE_COVERAGE_REPORT")
CANDIDATE_EVIDENCE_PATH = os.environ.get("HUMAN_CANDIDATE_EVIDENCE")


In [ ]:
COVERAGE_REPORT = None
if RUN_COVERAGE_FULL:
    COVERAGE_REPORT = run_coverage(COVERAGE_REPORT_PATH, CANDIDATE_EVIDENCE_PATH)
    print(COVERAGE_REPORT["analysisStatus"])
else:
    print("Coverage Full skipped. Real Train/Validation payloads were not opened.")


In [ ]:
if COVERAGE_REPORT is not None:
    show_coverage(COVERAGE_REPORT)
else:
    print("No Full coverage result. Internal fixtures are not Human dataset evidence.")


## 5. 저장 결과만 읽기

재학습/후보 재생성 없이 report를 읽습니다. `REPORT_TO_READ`에 저장한 report 경로를 넣고, `EVIDENCE_TO_CHECK`에는 함께 게시된 evidence를 지정합니다. 현재 코드·report에 기록된 dataset identity와 evidence SHA를 검증합니다. 원본 dataset checksum은 이 읽기 단계에서 다시 계산하지 않습니다. Dataset payload는 다시 열지 않습니다.


In [ ]:
REPORT_TO_READ = None
EVIDENCE_TO_CHECK = None
SAVED_REPORT = None
if REPORT_TO_READ is not None:
    require(isinstance(EVIDENCE_TO_CHECK, str) and EVIDENCE_TO_CHECK, "EVIDENCE_PATH_REQUIRED")
    source = Path(REPORT_TO_READ)
    evidence = Path(EVIDENCE_TO_CHECK)
    source = source if source.is_absolute() else ROOT / source
    evidence = evidence if evidence.is_absolute() else ROOT / evidence
    SAVED_REPORT = json.loads(source.read_text(encoding="utf-8"), parse_constant=reject_constant,
                              parse_float=finite_float)
    check_coverage_report(SAVED_REPORT)
    require(sha256_file(evidence) == SAVED_REPORT["evidence"]["sha256"], "EVIDENCE_SHA_MISMATCH")
    show_coverage(SAVED_REPORT)
else:
    print("Set REPORT_TO_READ and EVIDENCE_TO_CHECK to display saved Full results.")


## 6. 모델 비교 Full — 현재 구현하지 않음

Coverage 결과와 편향을 검토하고 학습 구간/예산을 승인한 뒤 구현합니다. 아래 스위치를 켜도 fit하지 않고 차단합니다. ID 원래 출력·같은 weight의 후보 mask·공유 scorer의 비교안은 명세에 있습니다. 모델 구현/Full 승인은 이 Notebook 작성과 별개입니다.


In [ ]:
RUN_MODEL_FULL = False
MODEL_REPORT = None
if RUN_MODEL_FULL:
    raise ValueError("COVERAGE_REVIEW_REQUIRED_MODEL_COMPARISON_NOT_IMPLEMENTED")
print("Model comparison blocked pending coverage review; modelFitCount=0.")


## 7. 해석·한계·다음 gate

- Target coverage는 정답이 선택지에 있는 비율이지 모델 정확도가 아닙니다. 후보가 많아지면 coverage가 올라가도 고르기는 더 어려울 수 있습니다.
- OOV는 기존 전체 Train 어휘 기준입니다. 미래 후보 적격 Train subset의 어휘와 같다고 가정하지 않습니다.
- 없는 target은 자동 불량 라벨이 아닙니다. 과거 geometry·관측 시점·정적 후보 계약의 한계일 수 있습니다. 원본/label/split/후보를 바꾸지 않습니다.
- Candidate1개는 선택 필요가 없고2개 이상은 선택이 필요합니다. 전체 구간·정답 포함·선택 필요 분모를 구분합니다.
- 다음은 사용자가 학습 없는 Coverage Full을 실행한 결과 해석입니다. 결과 검토 전 모델 fit·40행 통합·새 수집·commit은 자동 시작하지 않습니다.
